# Week 5-6 Exercises
Saebastion Cole

## Activity 5.01: Reading tabular data from a web page and creating dataframes

In [132]:
# Imports
import pandas as pd
# First i need to import the libraries, I am going to use BeautifulSoup to parse the saved copy of the page
from bs4 import BeautifulSoup
# SQLLite will be used for the db
import sqlite3

In [133]:
# This is the saved copy of the wikipedia page provided from the activity
# I don't plan on using html file but I saved it anyway
file_path = "data/List of countries by GDP (nominal) - Wikipedia.html"

with open(file_path, encoding="utf-8") as f:
    soup = BeautifulSoup(f, 'html.parser')

# The GDP by country table is the first table with the "wikitable" class
table = soup.find('table', class_='wikitable')
# This will get me the rows
rows = table.find_all('tr')

# Format a string for output I get the data count only when it is a td
print("Data rows (excluding header): " + str(len([r for r in rows if r.find('td')])))

Data rows (excluding header): 222


In [134]:
# The header row holds the source names, each header cell is a link with the source name followed by the year in brackets
header_cells = rows[0].find_all('th')
source_names = [cell.find('a').get_text(strip=True) for cell in header_cells[1:]]
print(f"Sources: {source_names}")

# Clean up the table by removing the <sup> tags
for sup in table.find_all('sup'):
    sup.decompose()

# Every row after the header holds a country name followed by one value per source
data_rows = [r for r in rows if r.find('td')]

# Build a separate dataframe for each source, pairing every country with that source value
source_data = {}
for i, source in enumerate(source_names, start=1):
    records = []
    for row in data_rows:
        cells = row.find_all('td')
        country = cells[0].get_text(' ', strip=True)
        value = cells[i].get_text(strip=True).replace(',', '')
        records.append({'Country': country, 'GDP (million US$)': value})
    df = pd.DataFrame(records)
    df['GDP (million US$)'] = pd.to_numeric(df['GDP (million US$)'], errors='coerce')
    source_data[source] = df

df.head()

Sources: ['IMF', 'World Bank', 'United Nations']


,Country,GDP (million US$)
0,World,111565144.0
1,United States,29298000.0
2,China,18743802.0
3,Germany,4659929.0
4,Japan,4026211.0


In [135]:
# The GDP by region table is the second table with the "wikitable" class
# This very closely follows what I already did for the previous data
region_table = soup.find_all('table', class_='wikitable')[1]
region_rows = region_table.find_all('tr')

print(f"Data rows (excluding header): {len([r for r in region_rows if r.find('td')])}")

# The header row holds the source names, same layout as the country table
region_header_cells = region_rows[0].find_all('th')
region_source_names = [cell.find('a').get_text(strip=True) for cell in region_header_cells[1:]]
print(f"Sources: {region_source_names}")

# Clean up the table by removing the <sup> tags
for sup in region_table.find_all('sup'):
    sup.decompose()

# Every row after the header holds a region name followed by one value per source
region_data_rows = [r for r in region_rows if r.find('td')]

# Build a separate dataframe for each source, pairing every region with that source value
region_source_data = {}
for i, source in enumerate(region_source_names, start=1):
    records = []
    for row in region_data_rows:
        cells = row.find_all('td')
        region = ' '.join(cells[0].get_text().split())
        value = cells[i].get_text(strip=True).replace(',', '')
        records.append({'Region': region, 'GDP (million US$)': value})
    df = pd.DataFrame(records)
    df['GDP (million US$)'] = pd.to_numeric(df['GDP (million US$)'], errors='coerce')
    region_source_data[source] = df

df.head()

Data rows (excluding header): 13
Sources: ['IMF', 'World Bank']


,Region,GDP (million US$)
0,World,110983000.0
1,Advanced Economies/High Income,71321000.0
2,G7,NaN
3,Emerging Market and Developing Economies,NaN
4,Emerging and Developing Asia,22261000.0


## Activity 6.01: Handling Outliers and Missing Data

In [136]:
# Read in the visit data
visit_df = pd.read_csv("data/visit_data.csv")
print("Rows, columns: " + str(visit_df.shape))
print(visit_df.head())

# Check for rows that are exact duplicates of another row
print("Duplicate rows: " + str(visit_df.duplicated().sum()))

# id and email should be unique per visitor, so check those columns on their own too
print("Duplicate ids: " + str(visit_df['id'].duplicated().sum()))
print("Duplicate emails: " + str(visit_df['email'].duplicated().sum()))

# Remove any duplicate rows, keeping the first occurrence
visit_df = visit_df.drop_duplicates()
print("Rows after removing duplicates: " + str(len(visit_df)))

Rows, columns: (1000, 7)
   id first_name last_name                       email gender  \
0   1      Sonny      Dahl            sdahl0@mysql.com   Male   
1   2        NaN       NaN           dhoovart1@hud.gov    NaN   
2   3        Gar     Armal      garmal2@technorati.com    NaN   
3   4    Chiarra     Nulty       cnulty3@newyorker.com    NaN   
4   5        NaN       NaN  sleaver4@elegantthemes.com    NaN   

        ip_address   visit  
0    135.36.96.183  1225.0  
1  237.165.194.143   919.0  
2   166.43.137.224   271.0  
3   139.98.137.108  1002.0  
4    46.117.117.27  2434.0  
Duplicate rows: 0
Duplicate ids: 0
Duplicate emails: 0
Rows after removing duplicates: 1000


In [137]:
# Check how many values are missing in each column
print("Missing values per column:")
print(visit_df.isna().sum())

# Dropping every row with any missing value would remove most of the data, since gender alone is missing in about half the rows
print("Rows with at least one missing value: " + str(visit_df.isna().any(axis=1).sum()))

# visit is the essential column for the analysis, so only remove the rows where it is missing
visit_df = visit_df.dropna(subset=['visit'])
print("Rows after removing records missing visit data: " + str(len(visit_df)))

Missing values per column:
id              0
first_name    296
last_name     296
email           0
gender        505
ip_address      0
visit          26
dtype: int64
Rows with at least one missing value: 663
Rows after removing records missing visit data: 974


## Inserting Data into a SQLite Database

In [138]:
import sqlite3

# Create the database in the data folder and a table for the contact information
con = sqlite3.connect("data/contacts.db")

# Drop the table first so rerunning this cell does not insert the rows twice
con.execute("DROP TABLE IF EXISTS contacts")

query = """
CREATE TABLE contacts
(Name VARCHAR(50), Address VARCHAR(100), City VARCHAR(50),
 State CHAR(2), Zip VARCHAR(10), Phone_Number VARCHAR(15)
);"""
con.execute(query)
con.commit()

# Made up contact data to insert into the db
data = [
    ("John Smith", "123 Main St", "Omaha", "NE", "68102", "402-555-0101"),
    ("Maria Garcia", "456 Oak Ave", "Austin", "TX", "78701", "512-555-0102"),
    ("David Johnson", "789 Pine Rd", "Denver", "CO", "80202", "303-555-0103"),
    ("Emily Davis", "321 Maple Dr", "Seattle", "WA", "98101", "206-555-0104"),
    ("Michael Brown", "654 Cedar Ln", "Chicago", "IL", "60601", "312-555-0105"),
    ("Sarah Wilson", "987 Birch Blvd", "Phoenix", "AZ", "85001", "602-555-0106"),
    ("James Miller", "147 Elm St", "Atlanta", "GA", "30303", "404-555-0107"),
    ("Linda Moore", "258 Spruce Ct", "Boston", "MA", "02108", "617-555-0108"),
    ("Robert Taylor", "369 Walnut Way", "Portland", "OR", "97201", "503-555-0109"),
    ("Jessica Anderson", "741 Willow Pl", "Nashville", "TN", "37201", "615-555-0110"),
    ("William Thomas", "852 Aspen Ave", "Lincoln", "NE", "68508", "402-555-0111"),
    ("Karen Jackson", "963 Poplar St", "Madison", "WI", "53703", "608-555-0112"),
]

# This creates the parameterized insert statement
stmt = "INSERT INTO contacts VALUES(?, ?, ?, ?, ?, ?)"

# This will execute it with all parameters passed
con.executemany(stmt, data)

# In databases commiting is the same as saving changes
con.commit()

# Query the table and load the results into a dataframe, using the cursor description for the column names
cursor = con.execute("SELECT * FROM contacts")
contact_rows = cursor.fetchall()
contacts_df = pd.DataFrame(contact_rows, columns=[x[0] for x in cursor.description])
print(contacts_df)

# Close the connection
con.close()

                Name         Address       City State    Zip  Phone_Number
0         John Smith     123 Main St      Omaha    NE  68102  402-555-0101
1       Maria Garcia     456 Oak Ave     Austin    TX  78701  512-555-0102
2      David Johnson     789 Pine Rd     Denver    CO  80202  303-555-0103
3        Emily Davis    321 Maple Dr    Seattle    WA  98101  206-555-0104
4      Michael Brown    654 Cedar Ln    Chicago    IL  60601  312-555-0105
5       Sarah Wilson  987 Birch Blvd    Phoenix    AZ  85001  602-555-0106
6       James Miller      147 Elm St    Atlanta    GA  30303  404-555-0107
7        Linda Moore   258 Spruce Ct     Boston    MA  02108  617-555-0108
8      Robert Taylor  369 Walnut Way   Portland    OR  97201  503-555-0109
9   Jessica Anderson   741 Willow Pl  Nashville    TN  37201  615-555-0110
10    William Thomas   852 Aspen Ave    Lincoln    NE  68508  402-555-0111
11     Karen Jackson   963 Poplar St    Madison    WI  53703  608-555-0112
